In [3]:
import os
import gzip
import requests
import pandas as pd
from tqdm.notebook import tqdm
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.PDB import PDBParser, DSSP
from Bio.PDB.Polypeptide import is_aa
os.environ["DSSP"] = "/Users/quineschuuring/miniforge3/bin/mkdssp"

In [25]:

# --- Load UniProt FASTA ---
fasta_file = "uniprotkb.fasta"

with gzip.open(fasta_file, "rt") as handle:
    records = list(SeqIO.parse(handle, "fasta"))

# --- Create DataFrame ---
data = {
    "ID": [rec.id for rec in records],
    "Sequence": [str(rec.seq) for rec in records],
    "Length": [len(rec.seq) for rec in records]
}

df = pd.DataFrame(data)

# --- Compute properties (MW, pI) ---
def compute_properties(seq):
    analysis = ProteinAnalysis(seq)
    try:
        mw = analysis.molecular_weight()
        pi = analysis.isoelectric_point()
        return pd.Series([mw, pi])
    except:
        return pd.Series([None, None])

df[['MW', 'pI']] = df['Sequence'].apply(compute_properties)


In [4]:
# --- Clean UniProt IDs ---
df["Accession"] = df["ID"].apply(lambda x: x.split("|")[1] if "|" in x else x)

# --- Download AlphaFold Structures ---
output_dir = "alphafold_structures"
os.makedirs(output_dir, exist_ok=True)

successful_downloads = []
failed_downloads = []

for uid in tqdm(df["Accession"], desc="Downloading AlphaFold PDBs"):
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb"
    out_path = os.path.join(output_dir, f"{uid}.pdb")

    if os.path.exists(out_path):
        successful_downloads.append(uid)
        continue

    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(out_path, 'wb') as f:
                f.write(response.content)
            successful_downloads.append(uid)
        else:
            failed_downloads.append(uid)
    except:
        failed_downloads.append(uid)

pd.Series(successful_downloads, name="Accession").to_csv("successful_downloads.csv", index=False)
pd.Series(failed_downloads, name="Accession").to_csv("failed_downloads.csv", index=False)


In [26]:
# --- Extract accession from ID ---
df["Accession"] = df["ID"].apply(lambda x: x.split("|")[1] if "|" in x else x)

# --- Load AlphaFold success/fail CSVs ---
success = pd.read_csv("successful_downloads.csv")
fail = pd.read_csv("failed_downloads.csv")

success["AlphaFold"] = True
fail["AlphaFold"] = False

all_af = pd.concat([success, fail], ignore_index=True)

# --- Merge with full UniProt df ---
df = df.merge(all_af, on="Accession", how="left")


In [10]:
import signal

# Set DSSP binary path
os.environ["DSSP"] = "/Users/quineschuuring/miniforge3/bin/mkdssp"

# DSSP wrapper with timeout
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

signal.signal(signal.SIGALRM, timeout_handler)

# Simplify SS codes
def simplify_dssp_code(code):
    if code in {'H', 'G', 'I'}:
        return 'H'
    elif code in {'E', 'B'}:
        return 'E'
    else:
        return 'C'

# Prepare lists
parser = PDBParser(QUIET=True)
secondary_structures = []
dssp_failures = []

test_df = df.sample(10, random_state=42)  # or use df.head(10)
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test DSSP"):

    pdb_path = os.path.join(output_dir, f"{row['Accession']}.pdb")

    if not os.path.exists(pdb_path):
        secondary_structures.append(None)
        dssp_failures.append(row['Accession'])
        continue

    try:
        signal.alarm(10)  # Timeout after 10 seconds
        structure = parser.get_structure(row["Accession"], pdb_path)
        model = structure[0]
        dssp = DSSP(model, pdb_path, dssp=os.environ["DSSP"])
        ss_sequence = "".join([
            simplify_dssp_code(dssp[key][2])
            for key in dssp.keys() if is_aa(dssp[key][1])
        ])
        secondary_structures.append(ss_sequence)
        signal.alarm(0)  # Clear timeout
    except TimeoutException:
        dssp_failures.append(row['Accession'])
        secondary_structures.append(None)
        print(f"⏳ Timeout: {row['Accession']}")
    except Exception as e:
        dssp_failures.append(row['Accession'])
        secondary_structures.append(None)
        signal.alarm(0)


# Assign results to the test_df only
test_df["SecondaryStructure"] = secondary_structures

# Optionally, print or inspect the result
print(test_df[["Accession", "SecondaryStructure"]])

# Save results if you want to verify
test_df.to_csv("test_secondary_structure.csv", index=False)

# # Add to df
# df["SecondaryStructure"] = secondary_structures
# pd.Series(dssp_failures, name="Accession").to_csv("dssp_failures.csv", index=False)
# df.to_csv("uniprot_with_secondary.csv", index=False)

print("✅ DSSP processing complete (with timeout protection).")


Test DSSP:   0%|          | 0/10 [00:00<?, ?it/s]

      Accession SecondaryStructure
9703     Q8WU08                   
16234    Q9HBL6                   
19034    B4E2M5                   
15063    Q03701                   
5160     P78385                   
993      O60493                   
15732    Q8NDH2                   
10280    Q96BF6                   
8748     Q8IY49                   
7223     Q5T7P3                   
✅ DSSP processing complete (with timeout protection).


In [17]:
import signal
from Bio.PDB import PDBParser, DSSP
from Bio.PDB.Polypeptide import is_aa

# Set DSSP binary path
os.environ["DSSP"] = "/Users/quineschuuring/miniforge3/bin/mkdssp"

# Timeout handler
class TimeoutException(Exception): pass
def timeout_handler(signum, frame): raise TimeoutException
signal.signal(signal.SIGALRM, timeout_handler)

# Simplify DSSP code
def simplify_dssp_code(code):
    if code in {'H', 'G', 'I'}: return 'H'
    elif code in {'E', 'B'}: return 'E'
    else: return 'C'

# --- Choose df for testing or full run ---
test_df = df.sample(10, random_state=42).copy()  # Uncomment for test run
working_df = test_df
# working_df = df  # Full run

# Setup
parser = PDBParser(QUIET=True)
secondary_structures = []
dssp_failures = []

for idx, row in tqdm(working_df.iterrows(), total=len(working_df), desc="Running DSSP (with timeout)"):
    pdb_path = os.path.join(output_dir, f"{row['Accession']}.pdb")

    if not os.path.exists(pdb_path):
        secondary_structures.append(None)
        dssp_failures.append(row['Accession'])
        continue

    try:
        signal.alarm(10)
        structure = parser.get_structure(row['Accession'], pdb_path)
        model = structure[0]
        dssp = DSSP(model, pdb_path, dssp=os.environ["DSSP"])
        ss_sequence = "".join([
            simplify_dssp_code(dssp[key][2])
            for key in dssp.keys() 
        ])
        
        secondary_structures.append(ss_sequence)
        signal.alarm(0)
    except TimeoutException:
        secondary_structures.append(None)
        dssp_failures.append(row['Accession'])
        print(f"⏳ Timeout: {row['Accession']}")
    except Exception:
        secondary_structures.append(None)
        dssp_failures.append(row['Accession'])
        signal.alarm(0)

# Add results (write only for full run)
df.loc[working_df.index, "SecondaryStructure"] = secondary_structures
if working_df is df:
    df.to_csv("uniprot_with_secondary.csv", index=False)
    pd.Series(dssp_failures, name="Accession").to_csv("dssp_failures.csv", index=False)
    print("DSSP full processing completed.")
else:
    print("Test run complete. Results not written to disk.")

working_df.head(10)


Running DSSP (with timeout):   0%|          | 0/10 [00:00<?, ?it/s]

Test run complete. Results not written to disk.


,ID,Sequence,Length,MW,pI,Accession,AlphaFold_x,AlphaFold_y,AlphaFold,SecondaryStructure
9703,sp|Q8WU08|ST32A_HUMAN,MGANTSRKPPVFDENEDVNFDHFEILRAIGKGSFGKVCIVQKNDTK...,396,46368.6487,6.874869,Q8WU08,True,True,True,
16234,sp|Q9HBL6|LRTM1_HUMAN,MKGELLLFSSVIVLLQVVCSCPDKCYCQSSTNFVDCSQQGLAEIPS...,345,38170.6558,6.258507,Q9HBL6,True,True,True,
19034,sp|B4E2M5|ANR66_HUMAN,MELAKMSDMTKLHQAVAAGDYSLVKKILKKGLCDPNYKDVDWNDRT...,196,22026.1116,9.042158,B4E2M5,True,True,True,
15063,sp|Q03701|CEBPZ_HUMAN,MAAVKEPLEFHAKRPWRPEEAVEDPDEEDEDNTSEAENGFSLEEVL...,1054,120972.5063,5.654650,Q03701,True,True,True,
5160,sp|P78385|KRT83_HUMAN,MTCGFNSIGCGFRPGNFSCVSACGPRPSRCCITAAPYRGISCYRGL...,493,54194.8952,5.539949,P78385,True,True,True,
993,sp|O60493|SNX3_HUMAN,MAETVADTRRLITKPQNLNDAYGPPSNFLEIDVSNPQTVGVGRGRF...,162,18762.1186,8.708083,O60493,True,True,True,
15732,sp|Q8NDH2|CC168_HUMAN,MSKQYYSFKKGVGSGLEDNTFMTLWDFLESWIIQNDWVAIFFIILL...,7081,801902.5274,8.942426,Q8NDH2,True,True,True,
10280,sp|Q96BF6|NACC2_HUMAN,MSQMLHIEIPNFGNTVLGCLNEQRLLGLYCDVSIVVKGQAFKAHRA...,587,62835.8296,5.638223,Q96BF6,True,True,True,
8748,sp|Q8IY49|PAQRA_HUMAN,MFAPRLLDFQKTKYARFMNHRVPAHKRYQPTEYEHAANCATHAFWI...,270,31263.3370,8.971759,Q8IY49,True,True,True,
7223,sp|Q5T7P3|LCE1B_HUMAN,MSCQQNQQQCQPPPKCIPKCPPKCLTPRCPPKCPPKCPPVSSCCSV...,118,11626.1550,8.831411,Q5T7P3,True,True,True,


In [6]:
df.to_csv("uniprot_sequences_with_properties.csv", index=False)

In [31]:
# --- Config ---
test_mode = False  # ⬅️ Toggle this to False for full run
sample_size = 50

# --- Timeout setup ---
class TimeoutException(Exception): pass
def timeout_handler(signum, frame): raise TimeoutException
signal.signal(signal.SIGALRM, timeout_handler)

# --- Simplify DSSP code ---
def simplify_dssp_code(code):
    if code in {'H', 'G', 'I'}:
        return 'H'
    elif code in {'E', 'B'}:
        return 'E'
    else:
        return 'C'

# --- DSSP run ---
parser = PDBParser(QUIET=True)
secondary_structures = []
dssp_failures = []

output_dir = "alphafold_structures"
target_df = df.sample(sample_size, random_state=42) if test_mode else df

for idx, row in tqdm(target_df.iterrows(), total=len(target_df), desc="Running DSSP"):
    uid = row['Accession']
    pdb_path = os.path.join(output_dir, f"{uid}.pdb")

    if not os.path.exists(pdb_path):
        secondary_structures.append(None)
        dssp_failures.append(uid)
        continue

    try:
        signal.alarm(10)
        structure = parser.get_structure(uid, pdb_path)
        model = structure[0]
        dssp = DSSP(model, pdb_path, dssp=os.environ["DSSP"])
        ss = "".join([
            simplify_dssp_code(dssp[key][2])
            for key in dssp.keys()
        ])
        secondary_structures.append(ss)
        signal.alarm(0)
    except Exception:
        secondary_structures.append(None)
        dssp_failures.append(uid)
        signal.alarm(0)

# --- Save or assign results ---
if test_mode:
    target_df["SecondaryStructure"] = secondary_structures
    print("\n🔍 TEST MODE: Showing last 50 entries with DSSP results:")
    display(target_df.tail(50))  # if in notebook, otherwise use print
else:
    df["SecondaryStructure"] = secondary_structures
    pd.Series(dssp_failures, name="Accession").to_csv("dssp_failures.csv", index=False)
    df.to_csv("uniprot_with_secondary.csv", index=False)
    print("✅ DSSP full run complete and saved.")


Running DSSP:   0%|          | 0/20418 [00:00<?, ?it/s]

✅ DSSP full run complete and saved.


df.tail(50)

In [32]:
df.tail(50)

,ID,Sequence,Length,MW,pI,Accession,AlphaFold,SecondaryStructure
20368,sp|Q96M42|CU129_HUMAN,MDGGSLRASPAAMDGGALEPAQQLSSLEGWTGQDRLLIPRWREARS...,142,15208.2094,8.277757,Q96M42,True,CCCCCCCCCCCCCCCCCCCHHHHHCCCCCCCCCCCCCCCCCCCCCC...
20369,sp|Q96M66|YP010_HUMAN,MAAKSTQDSLPRDTGEPSALPVQGRAEGRSSEGRKERTAECALRGK...,194,20692.9611,9.729262,Q96M66,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
20370,sp|Q96M78|FEAS2_HUMAN,MSQVGRVRSSHHFESVCLDAEVRVVLVALDHAGLHTLSSALNESLR...,137,15527.4951,6.439994,Q96M78,True,CCCCCCCCCCCCEEEEECCCCCEEEEECCCHHHHHHHHHHHHHHCC...
20371,sp|Q96M85|YV008_HUMAN,MSHSRRAAPTQDQCHTPGFPTSRETSGSIWQARICGSLQALDTWRT...,177,19601.0607,9.714564,Q96M85,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
20372,sp|Q96MF0|YO028_HUMAN,MPFKTKYPNGFHFAYLPTGSTQFRSLLQGQDSASQGVCPCRLCGAV...,132,14639.4715,8.820129,Q96MF0,True,CCCCEECCCCCEEEEEECCCHHHHHHHCCCCCCCCCCCHHHCECCC...
20373,sp|Q96MF4|CC140_HUMAN,MGDECSNPDLLAEPGSSPPWDHGNQRQEAANESNTRVPRVLKAHLG...,163,18252.2652,10.637944,Q96MF4,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHCCC...
20374,sp|Q96MH7|CE034_HUMAN,MAAELRMILYEDDSVQVQYVDGSTLQLSPCGSEFLFEKSPPVSAHP...,638,72883.2369,8.159393,Q96MH7,True,CCCEEEEEEECCCCEEEEECCCCEEEECCCCCEEEEECCCCCCCCC...
20375,sp|Q96MT0|YJ006_HUMAN,MFLHSGPARGPCTAAGRSASVRVPVQVAHELQGPDAIVFGAEVEQV...,163,16963.2038,5.291847,Q96MT0,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
20376,sp|Q96MT4|CF195_HUMAN,MIYPLDLFRNIPWKQGKCFASLSPEGERAFDGMEPLCQPGARPALR...,127,13895.6730,7.590131,Q96MT4,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
20377,sp|Q96N68|CR015_HUMAN,MQGQGALKESHIHLPTEQPEASLVLQGQLAESSALGPKGALRPQAQ...,181,19136.3771,7.447920,Q96N68,True,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
